In [1]:
n_components_list = [2, 3, 5] # number of components in each model we are going to use to compute WBIC and FE
# thermodynamic integration vars
ti_parallel_n_jobs=50
ti_parallel_n_verbose=5
ti_dcount=50

# wbic mcmc settings
n_tune=2500
n_draws=1000
n_chains=1

# wbic block settings
block_size=1
parallel_n_jobs=1
parallel_verbose=10

In [2]:
import os

n_trials=100
regimes = [50, 250, 5000]
print(f"Running on a machine with {os.cpu_count()} cpu cores")

Running on a machine with 128 cpu cores


In [3]:
import warnings
warnings.filterwarnings('ignore')

In [4]:
from pathlib import Path
from os import environ
import os

ROOT = Path(f"{Path.cwd().parents[1]}/outputs")
basedir = Path(f"{ROOT}/mixture/binom2d")
datadir = Path(f"{basedir}/data")
outputdir = Path(f"{basedir}/wbic-bias")

if not outputdir.exists():
  outputdir.mkdir(exist_ok=True)
  print(f"Created {outputdir}!")

print(f"Using datadir={datadir}")
print(f"Using outputdir={outputdir}")

dry = False # whether or not to save results

Using datadir=/mnt/zfs/jupyter-p03/home/a383ahme/waterloo-slt-reading-group/zoo/python/projects/luna/outputs/mixture/binom2d/data
Using outputdir=/mnt/zfs/jupyter-p03/home/a383ahme/waterloo-slt-reading-group/zoo/python/projects/luna/outputs/mixture/binom2d/wbic-bias


In [5]:
import pandas as pd

dgps_file = f"{datadir}/dgp.csv"
dgps=pd.read_csv(dgps_file, index_col=0).reset_index()
dgps.head()

,dsid,p0,p1,w0,w1
0,regular,0.15,0.35,0.75,0.25
1,e-singular,0.25,0.35,0.75,0.25
2,singular1,0.15,0.35,1.00,0.00
3,singular2,0.35,0.35,0.75,0.25


In [6]:
# from sklearn_extensions.mixbinom import BinomialMixture
import numpy as np


def find_truth_by_dsid(dsid: str):
  dgp = dgps.query(f"dsid=='{dsid}'")
  truth = dgp[["p0", "p1", "w0", "w1"]].iloc[0].tolist()
  return truth

# def rlct_by_dsid(dsid: str):
#   truth = find_truth_by_dsid(dsid)
#   n_components = np.ceil(len(truth)/2)
#   rlct = None
#   match dsid:
#     case "regular" | "e-singular":
#       rlct = (n_components*2-1)/2
#     case "singular1" | "singular2":
#       rlct = 1
#     case _:
#       raise Exception(f"Uknown dsid={dsid}")

#   return rlct

# def approx_free_energy_by_dsid(dsid, n_trials, X):
#   n=len(X)
  
#   average_log_likelihood = None
#   model = BinomialMixture(n_components=2, n_trials=n_trials, enforce_ordering=False)
#   input_data = np.column_stack([X, np.full_like(X, n_trials)])
#   model.fit(input_data)
#   mle, _ = model.point_estimate()
#   log_p = mixbinom.logpmf(weights=[mle[2], 1-mle[2]], probs=[mle[0], mle[1]], n=n_trials, x=X) # sample likelihood under the mle
#   average_log_likelihood = log_p.mean()

#   afe = -n*average_log_likelihood+rlct_by_dsid(dsid)*np.log(n)
#   return afe


# def expected_free_energy_by_dsid(dsid, n_trials, n):
#   X = np.arange(0, n_trials + 1)  # 0 to n_trials inclusive
#   truth = find_truth_by_dsid(dsid)
  
#   probs = truth[0:2]    # [p0, p1]
#   weights = truth[2:4]  # [w0, w1]
  
#   log_q = mixbinom.logpmf(n=n_trials, weights=weights, probs=probs, x=X)
  
#   expected_log_likelihood = np.sum(np.exp(log_q) * log_q)
  
#   second_order_term = rlct_by_dsid(dsid) * np.log(n)
#   efe = -n * expected_log_likelihood + second_order_term
  
#   return efe

In [7]:
# compute AFE and WBIC for many samples, 
# AFE requires sampling with beta=1 while WBIC requires sampling with beta=1/sqrt(n)
# for different n

In [8]:
import pandas as pd
import json
from pathlib import Path

fe_data_filename = "fe_by_ti"
fe_data_file = Path(f"{outputdir}/{fe_data_filename}.csv")
if dry:
  fe_data_file = Path(f"{fe_data_file}.dry")

# Load existing results if file exists
if fe_data_file.exists():
  fe_df = pd.read_csv(fe_data_file)
  # Create a set of completed (run, regime, dsid) tuples for fast lookup
  completed = set(
    zip(fe_df["trial"], fe_df["regime"], fe_df["dsid"], fe_df["n_components"])
  )
  fe_data = fe_df.to_dict("records")
else:
  completed = set()
  fe_data = []

def save_results():
  pd.DataFrame(fe_data).to_csv(fe_data_file, index=False)

In [ ]:
from joblib import Parallel, delayed
from tqdm import tqdm
from itertools import product
import time
from pymc_extensions.tempered_mixbinom import TemperedBinomialMixture, free_energy, free_energy_parallel
from pymc_extensions import pmx
from scipy_extensions import mixbinom
from tqdm.notebook import tqdm
import pymc as pm
import numpy as np
import arviz as az

def run_single_dsid(dsid, n_components, run, regime, datadir, n_trials, n_draws, n_tune, n_chains):
  dataset = pd.read_csv(f"{datadir}/{dsid}-{regime}.csv")
  X = dataset.iloc[:, run].to_numpy()
  n_obs = len(X)
  
  fe = free_energy_parallel(n_trials=n_trials, 
                            X=X,
                            betas=np.linspace(0,1,ti_dcount)**2,
                            n_components=n_components,
                            parallel_n_jobs=ti_parallel_n_jobs,
                            parallel_n_verbose=ti_parallel_n_verbose,
                            nuts_sampler="nutpie" if not dry else "numpyro")
  
  with TemperedBinomialMixture(X=X, n_trials=n_trials, beta=1/np.log(n_obs), n_components=n_components) as model:
    idata = model.sample(
      draws=n_draws,
      tune=n_tune,
      chains=n_chains,
      progressbar=False,
      nuts_sampler="nutpie" if not dry else "numpyro",
      cores=1
    )
    
    diverging = idata.sample_stats.diverging.values
    divs_per_chain = diverging.sum(axis=1)

    wbic = model.wbic(idata)

    print(f"run={run}, n_components={n_components}, regime={regime}, dsid={dsid}, wbic={wbic:.4f}, fe={fe:.4f}")
    
    return {
      "dsid": dsid,
      "regime": regime,
      "n": regime,
      "trial": run,
      "wbic": wbic,
      "fe": fe,
      "chains": n_chains,
      "draws": n_draws,
      "tune": n_tune,
      "mean_divergences": divs_per_chain.mean(),
      "total_divergences": diverging.sum(),
      "max_divergences": divs_per_chain.max(),
      "divergences_per_chain": divs_per_chain.tolist(),
      "chain_tree_depth": idata.sample_stats.depth.values.max(),
      "n_components": n_components
    }

# n_cores = os.cpu_count()
# All dsids
all_dsids = dgps["dsid"].unique()

# Group tasks by regime
# if n_cores > len(all_dsids):
#   block_size = int(n_cores/len(all_dsids)-1)
# else:
# block_size = 1

# Main loop - one run at a time, parallelize all (regime, dsid) combos
for block_start in tqdm(range(0, 1000, block_size), desc="blocks"):
  block_runs = range(block_start, min(block_start + block_size, 1000))
  for regime in regimes:
    for dsid in all_dsids:
      for n_components in n_components_list:
        start = time.perf_counter()
        # Build list of all (regime, dsid) pairs that haven't been completed
        tasks_to_run = [
          (run) 
          for run in block_runs
          if (run, regime, dsid, n_components) not in completed
        ]

        if not tasks_to_run:
          print(f"skipped {tasks_to_run}")
          continue

        print(f"about to run {tasks_to_run}, regime={regime}, dsid={dsid}, n_components={n_components}")

        # Run all regime x dsid combinations in parallel
        results = Parallel(n_jobs=parallel_n_jobs, verbose=parallel_verbose)(
          delayed(run_single_dsid)(
            dsid, n_components, run, regime, datadir, n_trials, n_draws, n_tune, n_chains
          )
          for run in tasks_to_run
        )

        # Collect results
        for result in results:
          fe_data.append(result)
          completed.add((result["trial"], result["regime"], result["dsid"], result["n_components"]))

        end = time.perf_counter()
        # Save after each run completes
        save_results()
        # if not dry:
        #   !git add f"../../outputs/mixture/binom2d/wbic-bias/{fe_data_filename}.csv"
        #   !git commit -m f"run n_components={n_components} block_runs={block_runs} complete"
        #   !git push

blocks:   0%|          | 0/1000 [00:00<?, ?it/s]

about to run [(0, 'regular', 5), (0, 'e-singular', 5), (0, 'singular1', 5), (0, 'singular2', 5)]


[Parallel(n_jobs=10)]: Using backend LokyBackend with 10 concurrent workers.
